# Bootstrap Confidence Intervals for Zero-Shot Evaluation Results

In [ ]:
import numpy as np
import json
import os
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

## Configuration

In [ ]:
# ── Configurable paths ──
MODEL_BASE_DIR = "./output"  # Path to model output directory
out_dir = os.path.join(MODEL_BASE_DIR, "scaling_v2/out_multi_win_True_standard_grad_clip_1.0_lr_0.00022_min_lr_2.2e-05_top_perc_0_n_layer_8_n_head_8_n_embd_1024_rotary_embedding_True_use_xpos_False_agg_labels_False_reverse_temporal_decay_0.5_block_size_512")

conditions = ["COPD", "CHF", "Dementia", "Pancreatic Cancer", "Breast Cancer",
              "Prostate Cancer", "Heart Attack", "Knee Osteoarthritis", "Stroke"]
time_horizons = [730, 1825]
approach = "direct"

n_bootstraps = 1000
# "stratified" = bootstrap positive and negative classes separately
# "standard"   = bootstrap all samples together
bootstrap_type = "stratified"

confidence_level = 0.95
random_seed = 42

# Parallelization: -1 = use all available cores, 1 = sequential (no parallelism)
n_jobs = -1

## Helper Functions

In [ ]:
def load_results(out_dir, condition_name, time_horizon, approach):
    """Load scores_and_labels.npz and metrics.json for a given condition/horizon."""
    results_dir = os.path.join(out_dir, f"{condition_name}_evaluation_results_{time_horizon}_{approach}")
    if not os.path.exists(results_dir):
        return None, None

    npz_path = os.path.join(results_dir, "scores_and_labels.npz")
    metrics_path = os.path.join(results_dir, "metrics.json")

    data = np.load(npz_path)
    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    return data, metrics


def compute_metrics(labels, scores, threshold):
    """Compute all metrics given labels, continuous scores, and a threshold."""
    preds = (scores >= threshold).astype(int)
    results = {}
    results["precision"] = precision_score(labels, preds, zero_division=0)
    results["recall"] = recall_score(labels, preds, zero_division=0)
    results["f1"] = f1_score(labels, preds, zero_division=0)

    # AUROC and AUPRC need both classes present
    if len(np.unique(labels)) > 1:
        results["auroc"] = roc_auc_score(labels, scores)
        results["auprc"] = average_precision_score(labels, scores)
    else:
        results["auroc"] = np.nan
        results["auprc"] = np.nan

    return results


def _single_bootstrap(labels, scores, threshold, pos_idx, neg_idx, bootstrap_type, seed):
    """Run a single bootstrap iteration. Designed to be called in parallel."""
    rng = np.random.RandomState(seed)
    if bootstrap_type == "stratified":
        boot_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        boot_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        boot_idx = np.concatenate([boot_pos, boot_neg])
    else:
        boot_idx = rng.choice(len(labels), size=len(labels), replace=True)

    return compute_metrics(labels[boot_idx], scores[boot_idx], threshold)


def bootstrap_ci(labels, scores, threshold, n_bootstraps=1000,
                 bootstrap_type="stratified", confidence_level=0.95,
                 random_seed=42, n_jobs=-1):
    """
    Compute bootstrap confidence intervals for all metrics.

    Parameters
    ----------
    labels : np.ndarray
        Binary ground truth labels.
    scores : np.ndarray
        Continuous predicted scores.
    threshold : float
        Optimal threshold (from validation set).
    n_bootstraps : int
        Number of bootstrap iterations.
    bootstrap_type : str
        "stratified" - resample positive and negative classes separately,
                       preserving the class ratio in each bootstrap sample.
        "standard"   - resample all samples together (class ratio may vary).
    confidence_level : float
        Confidence level for the interval (e.g. 0.95 for 95% CI).
    random_seed : int
        Random seed for reproducibility.
    n_jobs : int
        Number of parallel jobs for joblib (-1 = all cores, 1 = sequential).

    Returns
    -------
    dict: For each metric, a dict with "mean", "lower", "upper", "std", and "point_estimate".
    """
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    # Generate independent seeds for each bootstrap to ensure reproducibility
    rng = np.random.RandomState(random_seed)
    seeds = rng.randint(0, 2**31, size=n_bootstraps)

    # Run bootstraps in parallel
    boot_metrics_list = Parallel(n_jobs=n_jobs, prefer="processes")(
        delayed(_single_bootstrap)(
            labels, scores, threshold, pos_idx, neg_idx, bootstrap_type, int(seed)
        )
        for seed in seeds
    )

    # Collect results
    metric_names = ["precision", "recall", "f1", "auroc", "auprc"]
    boot_results = {m: np.array([b[m] for b in boot_metrics_list]) for m in metric_names}

    # Compute point estimates on the full data
    point_estimates = compute_metrics(labels, scores, threshold)

    alpha = 1 - confidence_level
    ci_results = {}
    for m in metric_names:
        values = boot_results[m]
        values = values[~np.isnan(values)]
        ci_results[m] = {
            "point_estimate": point_estimates[m],
            "mean": np.mean(values) if len(values) > 0 else np.nan,
            "std": np.std(values) if len(values) > 0 else np.nan,
            "lower": np.percentile(values, 100 * alpha / 2) if len(values) > 0 else np.nan,
            "upper": np.percentile(values, 100 * (1 - alpha / 2)) if len(values) > 0 else np.nan,
        }

    return ci_results

In [ ]:
# Alternative model directories (uncomment as needed):
# out_dir = os.path.join(MODEL_BASE_DIR, "scaling")
out_dir = os.path.join(MODEL_BASE_DIR, "scaling_judy/out_multi_win_True_standard_grad_clip_1.0_lr_0.00022_min_lr_2.2e-05_top_perc_0_n_layer_64_n_head_64_n_embd_1024_rotary_embedding_True_use_xpos_False_agg_labels_False_reverse_temporal_decay_0.5_block_size_512_data_1.0")
# out_dir = os.path.join(MODEL_BASE_DIR, "scaling_v2/out_multi_win_True_standard_grad_clip_1.0_lr_0.00022_min_lr_2.2e-05_top_perc_0_n_layer_8_n_head_8_n_embd_1024_rotary_embedding_True_use_xpos_False_agg_labels_False_reverse_temporal_decay_0.75_block_size_512")


# conditions = ["COPD", "CHF", "Dementia", "Pancreatic Cancer", "Breast Cancer",
#               "Prostate Cancer", "Heart Attack", "Knee Osteoarthritis", "Stroke"]
conditions = ["COPD", "CHF", "Dementia", "Pancreatic Cancer",
              "Prostate Cancer", "Heart Attack", "Knee Osteoarthritis"]
time_horizons = [730, 1825]
approach = "direct"

n_bootstraps = 1000
# "stratified" = bootstrap positive and negative classes separately
# "standard"   = bootstrap all samples together
bootstrap_type = "stratified"

confidence_level = 0.95
random_seed = 42

# Parallelization: -1 = use all available cores, 1 = sequential (no parallelism)
n_jobs = -1

## Run Bootstrap CI for All Conditions and Time Horizons

In [ ]:
all_results = []

for condition in tqdm(conditions):
    for horizon in time_horizons:
        data, metrics = load_results(out_dir, condition, horizon, approach)
        if data is None:
            print(f"Skipping {condition} / {horizon}d - results not found")
            continue

        test_scores = data["test_scores"]
        test_labels = data["test_labels"]
        threshold = metrics["optimal_threshold"]

        print(f"Processing {condition} / {horizon}d  (n={len(test_labels)}, "
              f"pos={test_labels.sum()}, neg={len(test_labels)-test_labels.sum()}, "
              f"bootstrap_type={bootstrap_type}, n_jobs={n_jobs}) ...")

        ci = bootstrap_ci(
            test_labels, test_scores, threshold,
            n_bootstraps=n_bootstraps,
            bootstrap_type=bootstrap_type,
            confidence_level=confidence_level,
            random_seed=random_seed,
            n_jobs=n_jobs,
        )

        for metric_name, vals in ci.items():
            all_results.append({
                "condition": condition,
                "time_horizon": horizon,
                "metric": metric_name,
                "point_estimate": vals["point_estimate"],
                "ci_lower": vals["lower"],
                "ci_upper": vals["upper"],
                "boot_mean": vals["mean"],
                "boot_std": vals["std"],
            })

print("Done.")

## Results Table

In [ ]:
df = pd.DataFrame(all_results)
df["ci_str"] = df.apply(lambda r: f"{r['point_estimate']:.4f} [{r['ci_lower']:.4f}, {r['ci_upper']:.4f}]", axis=1)

# Pivot for a nice display: rows = (condition, time_horizon), columns = metric
pivot = df.pivot_table(index=["condition", "time_horizon"], columns="metric", values="ci_str", aggfunc="first")
pivot = pivot[["auroc", "auprc", "f1", "precision", "recall"]]
pivot

In [ ]:
# save pivot table to CSV
pivot.to_csv("eval_data/bootstrap_ci_results_144M_0.5_data_1.0.csv")

## Raw DataFrame (for further analysis / export)

In [ ]:
df[["condition", "time_horizon", "metric", "point_estimate", "ci_lower", "ci_upper", "boot_std"]]